# CSV to FAIRFluids Conversion Example - DES Storage Experiment

This notebook demonstrates how to convert CSV data to FAIRFluids format using the updated `data_from_csv` method with the new DES Storage experiment format.

The `data_from_csv` method now supports:
- **New CSV format**: Semikolon-separated CSV with Compound 1-3, Molar Fraction 1-3, Property, Property Name, etc.
- **Multiple property types**: Viscosity and Polarity (and more) can be in the same fluid
- **Time parameter**: Instead of temperature, uses time (in days) as a parameter
- **Storage conditions**: Automatically extracts and maps storage conditions (none, closed, fridge, open)
- **Uncertainty handling**: Extracts standard deviation as uncertainty values
- **Automatic format detection**: Detects semikolon vs comma delimiters automatically

## Key Features
- **Compound combination grouping**: Each unique set of compounds gets its own document/JSON file
- **Multiple properties per fluid**: Different property types (Viscosity, Polarity) can be in the same fluid as separate measurements
- **Composition-based grouping**: Each unique composition (molar fractions + storage condition) creates a separate fluid
- **Storage information**: Storage conditions are stored at fluid level when consistent
- **Time-based measurements**: Tracks measurements over time (0, 7, 14, 38, 40 days)
- **Automatic parameter creation**: Mole fractions and time are automatically extracted as parameters


## Step 1: Import Required Libraries


In [1]:
import sys
import os
import pandas as pd
import json
from pathlib import Path

# Add parent directory to path (adjust to your path)
# sys.path.append('/home/YOURPATH/Code/FAIRFluids')

from fairfluids.core.fluid_io_2 import FluidIO
from fairfluids.core.lib import FAIRFluidsDocument, Version, StorageCondition


## Step 2: Load and Inspect the CSV Data


In [2]:
# Path to the CSV file
csv_path = '/home/sga/Code/FAIRFluids/fairfluids/data/DES_Storage_exp.csv'

# Load the CSV (semikolon-separated)
df = pd.read_csv(csv_path, sep=';')

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print("\nFirst few rows:")
df.head(10)


Total rows: 360
Columns: ['Compound 1', 'Compound 2', 'Compound 3', 'Molar Fraction 1', 'Molar Fraction 2', 'Molar Fraction 3', 'Property', 'Property Name', 'Standard Deviation Viscosity', 'Parameter', 'Type', 'Storage', 'Comment']

First few rows:


,Compound 1,Compound 2,Compound 3,Molar Fraction 1,Molar Fraction 2,Molar Fraction 3,Property,Property Name,Standard Deviation Viscosity,Parameter,Type,Storage,Comment
0,Cholinchloride,Glycerol,Water,"0,333","0,667",0,386,Viscosity,"5,842935",0,Time,none,ChCl-Gly (1:2)_0% H2O
1,Cholinchloride,Glycerol,Water,"0,333","0,667",0,370,Viscosity,"6,105516",7,Time,closed,ChCl-Gly (1:2)_0% H2O
2,Cholinchloride,Glycerol,Water,"0,333","0,667",0,368,Viscosity,"0,372161",14,Time,closed,ChCl-Gly (1:2)_0% H2O
3,Cholinchloride,Glycerol,Water,"0,333","0,667",0,344,Viscosity,"9,170496",38,Time,closed,ChCl-Gly (1:2)_0% H2O
4,Cholinchloride,Glycerol,Water,"0,333","0,667",0,364,Viscosity,"7,294365",7,Time,fridge,ChCl-Gly (1:2)_0% H2O
5,Cholinchloride,Glycerol,Water,"0,333","0,667",0,346,Viscosity,"12,914",14,Time,fridge,ChCl-Gly (1:2)_0% H2O
6,Cholinchloride,Glycerol,Water,"0,333","0,667",0,352,Viscosity,"3,141481",38,Time,fridge,ChCl-Gly (1:2)_0% H2O
7,Cholinchloride,Glycerol,Water,"0,333","0,667",0,367,Viscosity,"3,721615",7,Time,open,ChCl-Gly (1:2)_0% H2O
8,Cholinchloride,Glycerol,Water,"0,333","0,667",0,349,Viscosity,"8,150336",14,Time,open,ChCl-Gly (1:2)_0% H2O
9,Cholinchloride,Glycerol,Water,"0,333","0,667",0,260,Viscosity,"59,95521",40,Time,open,ChCl-Gly (1:2)_0% H2O


## Step 3: Inspect the CSV Structure

Let's examine the new CSV format:
- **Compound columns**: Compound 1, Compound 2, Compound 3
- **Molar fractions**: Molar Fraction 1, Molar Fraction 2, Molar Fraction 3
- **Property values**: Property (the measured value)
- **Property types**: Property Name (Viscosity, Polarity)
- **Uncertainty**: Standard Deviation Viscosity
- **Time parameter**: Parameter (time in days), Type (Time)
- **Storage**: Storage condition (none, closed, fridge, open)
- **Comments**: Additional information


In [3]:
# Show some example compound combinations
print("Example compound combinations from CSV:")
print(f"Total rows: {len(df)}")

# Show unique compound combinations
compound_combos = df[['Compound 1', 'Compound 2', 'Compound 3']].drop_duplicates()
print(f"\nUnique compound combinations: {len(compound_combos)}")
print("\nFirst 10 combinations:")
compound_combos.head(10)

# Show property types
print(f"\nProperty types in CSV:")
print(df['Property Name'].value_counts())

# Show storage conditions
print(f"\nStorage conditions:")
print(df['Storage'].value_counts())

# Show time parameter values
print(f"\nTime parameter values (days):")
print(df['Parameter'].value_counts().sort_index())


Example compound combinations from CSV:
Total rows: 360

Unique compound combinations: 6

First 10 combinations:

Property types in CSV:
Property Name
Viscosity    180
Polarity     180
Name: count, dtype: int64

Storage conditions:
Storage
closed    108
fridge    108
open      108
none       36
Name: count, dtype: int64

Time parameter values (days):
Parameter
0      36
7     108
14    108
38     54
39     18
40     36
Name: count, dtype: int64


## Step 4: Create FAIRFluids Documents from CSV

Now we'll use the updated `data_from_csv` method to create FAIRFluids documents.
The method will:
- **Automatically detect** semikolon delimiter
- **Group by unique compound combinations** (Compound 1, Compound 2, Compound 3)
- **Create separate documents** for each combination
- **Split properties**: Each property type (Viscosity, Polarity) creates separate fluids
- **Extract storage conditions**: Maps storage to StorageCondition enum
- **Handle time parameter**: Uses time (days) instead of temperature
- **Include uncertainty**: Extracts standard deviation values
- **Save each document** as a separate JSON file


In [10]:
# Create outputs directory if it doesn't exist
output_dir = Path('outputs_des_storage')
output_dir.mkdir(exist_ok=True)

# Create FluidIO instance
fluid_io = FluidIO()

# Create documents from CSV (one per compound combination)
# Set fetch_from_pubchem=False for faster testing (set to True to fetch compound info from PubChem)
print("Creating FAIRFluids documents from CSV...")
print("This may take a moment...\n")

documents = fluid_io.data_from_csv(
    csv_path=csv_path,
    output_dir=str(output_dir),
    fetch_from_pubchem=True  # Set to True to fetch compound info from PubChem (slower)
)

print(f"\n✅ Created {len(documents)} documents (one per compound combination)")
print(f"✅ Saved JSON files to: {output_dir}")

Creating FAIRFluids documents from CSV...
This may take a moment...

Found 6 unique compound combinations

Fetching compound information from PubChem for 8 unique compounds...

Fetched data for 7 compounds

  Added compound: Glycerol -> Glycerol (CID: 753)
  Added compound: Water -> Water (CID: 962)
Saved document to: outputs_des_storage/Cholinchloride_Glycerol_Water.json
  Added compound: Ethylene glycol -> Ethylene Glycol (CID: 174)
  Added compound: Water -> Water (CID: 962)
Saved document to: outputs_des_storage/Cholinchloride_Ethylene_glycol_Water.json
  Added compound: Betaine -> Betaine (CID: 247)
  Added compound: Glycerol -> Glycerol (CID: 753)
  Added compound: Water -> Water (CID: 962)
Saved document to: outputs_des_storage/Betaine_Glycerol_Water.json
  Added compound: Betaine -> Betaine (CID: 247)
  Added compound: Ethylene glycol -> Ethylene Glycol (CID: 174)
  Added compound: Water -> Water (CID: 962)
Saved document to: outputs_des_storage/Betaine_Ethylene_glycol_Water.js

## Step 5: Load and Inspect Created Documents

In [11]:
# Load all created documents
documents = []
for doc_path in sorted(output_dir.glob('*.json')):
    with open(doc_path, 'r', encoding='utf-8') as f:
        doc = FAIRFluidsDocument.model_validate_json(f.read())
        documents.append(doc)

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
if documents:
    print(documents[0])

Loaded 6 documents

First document preview:
version=Version(versionMajor=1, versionMinor=0) citation=None compound=[Compound(compoundID='Betaine', pubChemID=247, commonName='Betaine', SELFIE=None, name_IUPAC='2-(trimethylazaniumyl)acetate', standard_InChI='InChI=1S/C5H11NO2/c1-6(2,3)4-5(7)8/h4H2,1-3H3', standard_InChI_key='KWIUHFFTVRNATP-UHFFFAOYSA-N', molar_weigth=117.15, smiles_code='C[N+](C)(C)CC(=O)[O-]', sigma_profile=None), Compound(compoundID='Ethylene Glycol', pubChemID=174, commonName='Ethylene Glycol', SELFIE=None, name_IUPAC='ethane-1,2-diol', standard_InChI='InChI=1S/C2H6O2/c3-1-2-4/h3-4H,1-2H2', standard_InChI_key='LYCAIKOWRPUZTN-UHFFFAOYSA-N', molar_weigth=62.07, smiles_code='C(CO)O', sigma_profile=None), Compound(compoundID='Water', pubChemID=962, commonName='Water', SELFIE=None, name_IUPAC='oxidane', standard_InChI='InChI=1S/H2O/h1H2', standard_InChI_key='XLYOFNOQVPJJNP-UHFFFAOYSA-N', molar_weigth=18.015, smiles_code='O', sigma_profile=None)] fluid=[Fluid(fluidID=['6e63

## Step 6: Inspect Document Structure

Let's examine one of the created documents to verify the structure, including:
- Storage conditions
- Time parameters
- Property types (Viscosity, Polarity)
- Uncertainty values


In [6]:
# Inspect the first document
if documents:
    doc = documents[0]
    
    print(f"Document 1 summary:")
    print(f"  - Compounds: {[c.compoundID for c in doc.compound]}")
    print(f"  - Number of fluids: {len(doc.fluid)}")
    
    for i, fluid in enumerate(doc.fluid):
        print(f"\n  Fluid {i+1}:")
        print(f"    Compounds: {fluid.compounds}")
        print(f"    Properties: {[p.propertyID for p in fluid.property]}")
        print(f"    Parameters: {[p.parameterID for p in fluid.parameter]}")
        print(f"    Storage: {fluid.storage.storage_condition.value if fluid.storage and fluid.storage.storage_condition else 'None'}")
        print(f"    Measurements: {len(fluid.measurement)}")
        
        # Show first measurement details
        if fluid.measurement:
            first_meas = fluid.measurement[0]
            print(f"    First measurement:")
            prop_val = first_meas.propertyValue[0] if first_meas.propertyValue else None
            if prop_val:
                print(f"      Property value: {prop_val.propValue}")
                print(f"      Uncertainty: {prop_val.uncertainty if prop_val.uncertainty else 'None'}")
            print(f"      Parameter values: {len(first_meas.parameterValue)} parameters")
            for pv in first_meas.parameterValue:
                print(f"        - {pv.parameterID}: {pv.paramValue}")
            if first_meas.method_description:
                print(f"      Method description: {first_meas.method_description}")
else:
    print("No documents were created.")


Document 1 summary:
  - Compounds: ['Betaine', 'Ethylene Glycol', 'Water']
  - Number of fluids: 12

  Fluid 1:
    Compounds: ['Betaine', 'Ethylene Glycol', 'Water']
    Properties: ['viscosity', 'polarity']
    Parameters: ['parameter_time', 'parameter_mole_fraction_betaine', 'parameter_mole_fraction_ethyleneglycol', 'parameter_mole_fraction_water']
    Storage: None
    Measurements: 2
    First measurement:
      Property value: 72.01
      Uncertainty: 1.126336
      Parameter values: 4 parameters
        - parameter_time: 0.0
        - parameter_mole_fraction_betaine: 0.25
        - parameter_mole_fraction_ethyleneglycol: 0.75
        - parameter_mole_fraction_water: 0.0
      Method description: Experimental measurement (Comment: Bet-EG (1:3)_0% H2O)

  Fluid 2:
    Compounds: ['Betaine', 'Ethylene Glycol', 'Water']
    Properties: ['viscosity', 'polarity']
    Parameters: ['parameter_time', 'parameter_mole_fraction_betaine', 'parameter_mole_fraction_ethyleneglycol', 'parameter_

## Step 7: Verify Document Structure

Let's verify that:
1. Each document groups fluids by composition (molar fractions + storage condition)
2. Fluids can contain multiple properties (Viscosity, Polarity) as separate measurements
3. Parameters include mole fractions and time (instead of temperature)
4. Storage conditions are properly set


In [7]:
# Verify structure across all documents
print("Verifying structure across all documents:\n")

fluids_with_multiple_properties = []
total_fluids = 0
fluids_with_viscosity = 0
fluids_with_polarity = 0
fluids_with_both_properties = 0
fluids_with_storage = 0
fluids_with_time_param = 0

for doc_idx, doc in enumerate(documents):
    for fluid in doc.fluid:
        total_fluids += 1
        
        # Check properties
        property_ids = [p.propertyID for p in fluid.property]
        has_viscosity = "viscosity" in property_ids
        has_polarity = "polarity" in property_ids
        
        if has_viscosity:
            fluids_with_viscosity += 1
        if has_polarity:
            fluids_with_polarity += 1
        if has_viscosity and has_polarity:
            fluids_with_both_properties += 1
        
        if len(fluid.property) > 1:
            fluids_with_multiple_properties.append((doc_idx, len(fluid.property), property_ids))
        
        # Check for storage
        if fluid.storage and fluid.storage.storage_condition:
            fluids_with_storage += 1
        
        # Check for time parameter
        if any(p.parameterID == "parameter_time" for p in fluid.parameter):
            fluids_with_time_param += 1

print(f"Total documents: {len(documents)}")
print(f"Total fluids: {total_fluids}")
print(f"  - Fluids with viscosity property: {fluids_with_viscosity}")
print(f"  - Fluids with polarity property: {fluids_with_polarity}")
print(f"  - Fluids with both properties: {fluids_with_both_properties}")
print(f"  - Fluids with storage condition: {fluids_with_storage}")
print(f"  - Fluids with time parameter: {fluids_with_time_param}")

if fluids_with_multiple_properties:
    print(f"\n✅ Found {len(fluids_with_multiple_properties)} fluids with multiple properties (as expected):")
    for doc_idx, prop_count, prop_ids in fluids_with_multiple_properties[:5]:
        print(f"  Document {doc_idx}: {prop_count} properties ({', '.join(prop_ids)})")
else:
    print("\n⚠️  No fluids with multiple properties found")


Verifying structure across all documents:

Total documents: 6
Total fluids: 72
  - Fluids with viscosity property: 72
  - Fluids with polarity property: 72
  - Fluids with both properties: 72
  - Fluids with storage condition: 54
  - Fluids with time parameter: 72

✅ Found 72 fluids with multiple properties (as expected):
  Document 0: 2 properties (viscosity, polarity)
  Document 0: 2 properties (viscosity, polarity)
  Document 0: 2 properties (viscosity, polarity)
  Document 0: 2 properties (viscosity, polarity)
  Document 0: 2 properties (viscosity, polarity)


## Step 8: Check Output Files

List the JSON files that were created in the outputs folder.


In [8]:
# List all JSON files created
json_files = list(output_dir.glob('*.json'))
print(f"Created {len(json_files)} JSON files:\n")

for json_file in sorted(json_files)[:10]:  # Show first 10
    size_kb = os.path.getsize(json_file) / 1024
    print(f"  {json_file.name}: {size_kb:.2f} KB")

if len(json_files) > 10:
    print(f"  ... and {len(json_files) - 10} more files")


Created 6 JSON files:

  Betaine_Ethylene_glycol_Water.json: 98.16 KB
  Betaine_Glycerol_Water.json: 97.63 KB
  Cholinchloride_Ethylene_glycol_Water.json: 85.72 KB
  Cholinchloride_Glycerol_Water.json: 98.33 KB
  Lidocaine_Oxalic_acid_Water.json: 93.64 KB
  Tetrabutylammonium_c_Glycerol_Water.json: 95.13 KB


## Step 9: Summary Statistics

Show summary statistics about the created documents, including property types, storage conditions, and time parameters.


In [9]:
# Summary statistics
total_measurements = 0
documents_with_viscosity = 0
documents_with_polarity = 0
measurements_with_uncertainty = 0
storage_conditions = {}
viscosity_measurements = 0
polarity_measurements = 0

for doc in documents:
    has_viscosity = any(
        any(p.propertyID == "viscosity" for p in f.property)
        for f in doc.fluid
    )
    has_polarity = any(
        any(p.propertyID == "polarity" for p in f.property)
        for f in doc.fluid
    )
    
    if has_viscosity:
        documents_with_viscosity += 1
    if has_polarity:
        documents_with_polarity += 1
    
    for fluid in doc.fluid:
        total_measurements += len(fluid.measurement)
        
        # Count storage conditions
        if fluid.storage and fluid.storage.storage_condition:
            storage = fluid.storage.storage_condition.value
            storage_conditions[storage] = storage_conditions.get(storage, 0) + 1
        
        # Count measurements with uncertainty and by property type
        for meas in fluid.measurement:
            if meas.propertyValue:
                prop_val = meas.propertyValue[0]
                if prop_val.uncertainty is not None:
                    measurements_with_uncertainty += 1
                if prop_val.propertyID == "viscosity":
                    viscosity_measurements += 1
                elif prop_val.propertyID == "polarity":
                    polarity_measurements += 1

print("Summary Statistics:")
print(f"  Total documents: {len(documents)}")
print(f"  Documents with viscosity: {documents_with_viscosity}")
print(f"  Documents with polarity: {documents_with_polarity}")
print(f"  Total fluids: {total_fluids}")
print(f"  Total measurements: {total_measurements}")
print(f"    - Viscosity measurements: {viscosity_measurements}")
print(f"    - Polarity measurements: {polarity_measurements}")
print(f"  Measurements with uncertainty: {measurements_with_uncertainty}")
print(f"  Average measurements per fluid: {total_measurements / total_fluids:.1f}" if total_fluids > 0 else "  Average measurements per fluid: 0")
if storage_conditions:
    print(f"\n  Storage conditions distribution:")
    for storage, count in sorted(storage_conditions.items()):
        print(f"    - {storage}: {count} fluids")

Summary Statistics:
  Total documents: 6
  Documents with viscosity: 6
  Documents with polarity: 6
  Total fluids: 72
  Total measurements: 340
    - Viscosity measurements: 170
    - Polarity measurements: 170
  Measurements with uncertainty: 334
  Average measurements per fluid: 4.7

  Storage conditions distribution:
    - Closed: 18 fluids
    - Fridge: 18 fluids
    - Open: 18 fluids
